# Introduction to Graphs with LangGraph

LangGraph is based on [Pregel: a system for large-scale graph processing](https://15799.courses.cs.cmu.edu/fall2013/static/papers/p135-malewicz.pdf).

In the following we will create a simple state graph using the LangGraph library. The graph will represent a sequence of steps to collect user information, including first name, last name, age, and email. Note at this point, that the workflow we are creating is neither an agentic workflow nor an agent, as we do not have any LLM calls in it.

In [ ]:
import random
from pydantic import BaseModel, Field

Pydantic is used to define the state model, which holds the user information. Pydantic is a good choice for defining data models in Python due to its ease of use, data validation capabilities, and integration with type hints.

In [ ]:
class State(BaseModel):
    """State for the user profile generation."""

    first_name: str = Field(None, description="The first name of the user.")
    last_name: str = Field(None, description="The last name of the user.")
    age: int = Field(None, description="The age of the user.")
    email: str = Field(None, description="The email of the user.")

The state is a collection of fields that we want to fill out. In general, the state will be updated as it flows through the graph nodes.

In LangGraph a graph is built using a builder pattern. We first create a builder, then add nodes and edges to it, and finally compile the graph. Nodes represent the individual steps or actions in the workflow implemented as Python functions. Edges define the transitions between these nodes, specifying the flow and direction of the process. This allows for directed workflows and workflows with branches and loops.

In [ ]:
# Create the node for the first name.
def set_first_name(state: State) -> State:
    """Set the first name in the state."""
    possibilities = ["Hans", "Peter", "Klaus", "Jürgen", "Michael"]
    first_name = random.choice(possibilities)
    return {
        "first_name": first_name
    }

# Create the node for the last name.
def set_last_name(state: State) -> State:
    """Set the last name in the state."""
    possibilities = ["Müller", "Schmidt", "Schneider", "Fischer", "Weber"]
    last_name = random.choice(possibilities)
    return {
        "last_name": last_name
    }

# Create the node for the age.
def set_age(state: State) -> State:
    """Set the age in the state."""
    possibilities = range(18, 100)
    age = random.choice(possibilities)
    return {
        "age": age
    }

# Create the node for the email.
def set_email(state: State) -> State:
    """Set the email in the state."""
    email = f"{state.first_name.lower()}.{state.last_name.lower()}@example.com"
    return {
        "email": email
    }

As you might have noticed, each node function takes the current state as input and returns an updated state. This is a common pattern in state graphs, where each node modifies the state based on its logic.

In [ ]:
from langgraph.graph import StateGraph, START, END

# Create the builder for the state graph.
builder = StateGraph(State)

# Add the nodes to the graph.
builder.add_node("set_first_name", set_first_name)
builder.add_node("set_last_name", set_last_name)
builder.add_node("set_age", set_age)
builder.add_node("set_email", set_email)

# Add the edges to the graph.
builder.add_edge(START, "set_first_name")
builder.add_edge("set_first_name", "set_last_name")
builder.add_edge("set_last_name", "set_age")
builder.add_edge("set_age", "set_email")
builder.add_edge("set_email", END)

# Create the state graph.
graph = builder.compile()

Now the graph is ready to be invoked with an initial state. The graph will process the state through its nodes, updating it at each step, and finally return the completed state. But first, let us visualize the graph:

In [ ]:
# Render the graph.
from IPython.display import display
display(graph)

This is a fine sequential workflow. How would we turn this into an agentic workflow or an agent? We would need to add LLM calls into the nodes, and possibly also add some branching logic based on the state or LLM outputs. This could involve using decision nodes that evaluate conditions and direct the flow accordingly, or incorporating nodes that generate prompts for the LLM based on the current state. By integrating these elements, we can create a more dynamic and interactive workflow that leverages the capabilities of language models. We will do this later.

Let us run the graph now:

In [ ]:
# Invoke the state graph with an initial state.
result = graph.invoke(State())
print(result)

## Branching out.

Low let us consider how we can run things in parallel.

In [ ]:
builder = StateGraph(State)

builder.add_node(set_first_name)
builder.add_node(set_last_name)
builder.add_node(set_age)
builder.add_node(set_email)

builder.add_edge(START, "set_first_name")
builder.add_edge(START, "set_last_name")
builder.add_edge(START, "set_age")
builder.add_edge("set_first_name", "set_email")
builder.add_edge("set_last_name", "set_email")
builder.add_edge("set_age", "set_email")
builder.add_edge("set_email", END)

graph = builder.compile()

In [ ]:
display(graph)

In [ ]:
# Invoke the state graph with an initial state.
result = graph.invoke(State())
print(result)

# Done.